In [5]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.
simulator = BasicSimulator()

def run_one_shot(qc):
    tqc = transpile(qc, simulator)
    result = simulator.run(tqc, shots=1, memory=True).result()
    return result.get_memory()[0]

def bit_to_pm_one(bit):
    return 1 if bit == 0 else -1

def choice_gate_one_third():
    a = math.sqrt(1/3)
    b = math.sqrt(2/3)

    qc = QuantumCircuit(1, name="U_1/3")
    qc.unitary(Operator([
        [a, -b],
        [b,  a]
    ]), [0])
    return qc.to_gate()

def quantum_random_choice_1_to_3():

    first = QuantumCircuit(1, 1)
    first.append(choice_gate_one_third(), [0])
    first.measure(0, 0)

    first_result = int(run_one_shot(first))

    if first_result == 0:
        return 1

    second = QuantumCircuit(1, 1)
    second.h(0)
    second.measure(0, 0)

    second_result = int(run_one_shot(second))

    if second_result == 0:
        return 2
    else:
        return 3

def prepare_singlet(qc):
    """
    Prepare (|01> - |10>) / sqrt(2) on qubits 0 and 1.
    """
    qc.x(1)
    qc.h(0)
    qc.cx(0, 1)
    qc.z(0)


def apply_measurement_basis(qc, qubit, basis):
    if basis == 'Z':
        pass
    elif basis == 'X':
        qc.h(qubit)
    elif basis == 'W':
        qc.ry(-math.pi / 4, qubit)
    elif basis == 'V':
        qc.ry(math.pi / 4, qubit)
    else:
        raise ValueError(f"Unknown basis: {basis}")

def alice_basis(choice):
    if choice == 1:
        return 'X'
    elif choice == 2:
        return 'W'
    elif choice == 3:
        return 'Z'
    else:
        raise ValueError("Alice choice must be 1, 2 or 3")


def bob_basis(choice):
    if choice == 1:
        return 'W'
    elif choice == 2:
        return 'Z'
    elif choice == 3:
        return 'V'
    else:
        raise ValueError("Bob choice must be 1, 2 or 3")

def measure_entangled_pair(alice_choice, bob_choice):

    a_basis = alice_basis(alice_choice)
    b_basis = bob_basis(bob_choice)

    qc = QuantumCircuit(2, 2)

    prepare_singlet(qc)

    apply_measurement_basis(qc, 0, a_basis)
    apply_measurement_basis(qc, 1, b_basis)

    qc.measure(0, 0)
    qc.measure(1, 1)

    bitstring = run_one_shot(qc)

    # if c0 is Alice and c1 is Bob, memory string is usually "b1b0"
    alice_bit = int(bitstring[1])
    bob_bit = int(bitstring[0])

    return alice_bit, bob_bit

In [7]:
def run_e91_plain(N):
    rounds = math.ceil(9 * N / 2)

    alice_key = []
    bob_key = []

    # For Bell test totals
    sums = {
        "XW": 0,
        "XV": 0,
        "ZW": 0,
        "ZV": 0
    }

    counts = {
        "XW": 0,
        "XV": 0,
        "ZW": 0,
        "ZV": 0
    }

    discarded = 0

    all_results = []

    for _ in range(rounds):
        a_choice = quantum_random_choice_1_to_3()
        b_choice = quantum_random_choice_1_to_3()

        a_bit, b_bit = measure_entangled_pair(a_choice, b_choice)

        all_results.append((a_choice, b_choice, a_bit, b_bit))

        # Key-generation cases
        if (a_choice, b_choice) == (2, 1) or (a_choice, b_choice) == (3, 2):
            alice_key.append(a_bit)
            bob_key.append(1 - b_bit)   # Bob inverts because outcomes are opposite

        # Bell-test cases
        elif (a_choice, b_choice) == (1, 1):   # X,W
            sums["XW"] += bit_to_pm_one(a_bit) * bit_to_pm_one(b_bit)
            counts["XW"] += 1

        elif (a_choice, b_choice) == (1, 3):   # X,V
            sums["XV"] += bit_to_pm_one(a_bit) * bit_to_pm_one(b_bit)
            counts["XV"] += 1

        elif (a_choice, b_choice) == (3, 1):   # Z,W
            sums["ZW"] += bit_to_pm_one(a_bit) * bit_to_pm_one(b_bit)
            counts["ZW"] += 1

        elif (a_choice, b_choice) == (3, 3):   # Z,V
            sums["ZV"] += bit_to_pm_one(a_bit) * bit_to_pm_one(b_bit)
            counts["ZV"] += 1

        else:
            discarded += 1

    def average(label):
        if counts[label] == 0:
            return 0
        return sums[label] / counts[label]

    EXW = average("XW")
    EXV = average("XV")
    EZW = average("ZW")
    EZV = average("ZV")

    S = abs(EXW - EXV + EZW + EZV)

    return {
        "rounds": rounds,
        "alice_key": alice_key,
        "bob_key": bob_key,
        "key_match": alice_key == bob_key,
        "key_length": len(alice_key),
        "discarded_count": discarded,
        "counts": counts,
        "expectations": {
            "E(X,W)": EXW,
            "E(X,V)": EXV,
            "E(Z,W)": EZW,
            "E(Z,V)": EZV
        },
        "S": S,
        "all_results": all_results
    }

In [9]:
result = run_e91_plain(200)

print("Rounds run:", result["rounds"])
print("Key length produced:", result["key_length"])
print("Alice key:", result["alice_key"])
print("Bob key:  ", result["bob_key"])
print("Keys match:", result["key_match"])
print()
print("Bell test counts:", result["counts"])
print("Expectation values:")
for k, v in result["expectations"].items():
    print(f"  {k} = {v:.4f}")
print()
print("S =", result["S"])
print("2*sqrt(2) =", 2 * math.sqrt(2))

Rounds run: 900
Key length produced: 203
Alice key: [0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0]
Bob key:   [0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1,